# 準備演習 01: エスカレーションロジックを備えたマルチツールエージェント

## 目的

- Claude Agent SDK の `@tool` でカスタムツールを定義する方法を学ぶ
- `create_sdk_mcp_server()` で MCP サーバーにツールを登録する
- `ClaudeSDKClient` でエージェントを実行する
- `PreToolUse` / `PostToolUse` フックでビジネスルールを実装する
- 構造化エラー、リトライ、ビジネスルールガード、エスカレーションを実装する

## 対象ドメイン

- Domain 1: Agentic Architecture & Orchestration
- Domain 2: Tool Design & MCP Integration
- Domain 5: Context Management & Reliability

## 完成イメージ

この Notebook では **Claude Agent SDK を実際に使用した実装** を体験します。
最新の API・hooks・MCP server の書き方は次を参照してください。

- 公式: `https://platform.claude.com/docs/en/agent-sdk/overview`
- 公式: `https://platform.claude.com/docs/en/agent-sdk/hooks`
- 完成版 Lab: [../labs/01-support-agent/](../labs/01-support-agent/)

In [ ]:
from __future__ import annotations

import json
from typing import Any
from dotenv import load_dotenv

from claude_agent_sdk import ClaudeAgentOptions, ClaudeSDKClient, create_sdk_mcp_server, tool
from claude_agent_sdk.types import (
    AssistantMessage,
    HookContext,
    HookInput,
    HookJSONOutput,
    HookMatcher,
    ResultMessage,
    TextBlock,
    ToolUseBlock,
)

load_dotenv()

REFUND_THRESHOLD = 10000.0

def section(title: str):
    print(f"\n=== {title} ===")

## Step 1. セッション状態と擬似データベースを準備する

In [ ]:
# セッション状態 (プログラム的前提条件ゲートに使用)
session_state: dict[str, Any] = {
    "customer_verified": False,
    "verified_customer_id": None,
}

# 擬似データベース
CUSTOMERS_DB = {
    "CUST-001": {
        "id": "CUST-001",
        "name": "田中 太郎",
        "email": "tanaka@example.com",
        "is_verified": True,
    },
}

ORDERS_DB = {
    "ORD-001": {
        "id": "ORD-001",
        "customer_id": "CUST-001",
        "items": [{"name": "ノートPC", "price": 120000, "qty": 1}],
        "total": 120000,
        "status": "delivered",
        "can_refund": True,
    },
    "ORD-002": {
        "id": "ORD-002",
        "customer_id": "CUST-001",
        "items": [{"name": "マウス", "price": 3000, "qty": 2}],
        "total": 6000,
        "status": "delivered",
        "can_refund": True,
    },
}

section("セッション状態とデータベースの準備完了")
print(f"顧客数: {len(CUSTOMERS_DB)}")
print(f"注文数: {len(ORDERS_DB)}")

## Step 2. ビジネスロジック関数を定義する

ツール実装の前に、実際のビジネスロジックを関数として定義します。

In [ ]:
def handle_get_customer(customer_id: str) -> dict[str, Any]:
    """顧客情報を取得し、セッション状態を更新する"""
    customer = CUSTOMERS_DB.get(customer_id)
    if not customer:
        return {
            "isError": True,
            "errorCategory": "validation",
            "isRetryable": False,
            "content": [{"type": "text", "text": f"Customer {customer_id} not found"}],
        }
    
    # セッション状態を更新
    session_state["customer_verified"] = True
    session_state["verified_customer_id"] = customer_id
    
    return {
        "isError": False,
        "customer": customer,
    }


def handle_lookup_order(order_id: str) -> dict[str, Any]:
    """注文詳細を取得する"""
    order = ORDERS_DB.get(order_id)
    if not order:
        return {
            "isError": True,
            "errorCategory": "validation",
            "isRetryable": False,
            "content": [{"type": "text", "text": f"Order {order_id} not found"}],
        }
    
    return {
        "isError": False,
        "order": order,
    }


def handle_process_refund(order_id: str, amount: float, reason: str) -> dict[str, Any]:
    """返金処理を実行する (前提条件をチェック)"""
    # プログラム的前提条件ゲート
    if not session_state["customer_verified"]:
        return {
            "isError": True,
            "errorCategory": "validation",
            "isRetryable": True,
            "content": [{"type": "text", "text": "Customer verification required before refund"}],
            "userMessage": "先に get_customer で本人確認してください。",
        }
    
    order = ORDERS_DB.get(order_id)
    if not order:
        return {
            "isError": True,
            "errorCategory": "validation",
            "isRetryable": False,
            "content": [{"type": "text", "text": f"Order {order_id} not found"}],
        }
    
    if not order["can_refund"]:
        return {
            "isError": True,
            "errorCategory": "business_rule",
            "isRetryable": False,
            "content": [{"type": "text", "text": f"Order {order_id} cannot be refunded"}],
        }
    
    # 高額返金チェックは PreToolUse フックで行う
    return {
        "isError": False,
        "status": "approved",
        "order_id": order_id,
        "amount": amount,
        "reason": reason,
    }


def handle_escalate_to_human(reason: str, context: dict[str, Any]) -> dict[str, Any]:
    """人間へのエスカレーション"""
    return {
        "isError": False,
        "handoff": {
            "reason": reason,
            "context": context,
            "queue": "refund-approval",
        },
    }


def build_tool_result(payload: dict[str, Any], *, is_error: bool) -> dict[str, Any]:
    """ツール結果を Agent SDK フォーマットに整形する"""
    return {
        "content": [
            {"type": "text", "text": json.dumps(payload, ensure_ascii=False)}
        ],
        "is_error": is_error,
    }


section("ビジネスロジック関数の定義完了")
print("✓ handle_get_customer")
print("✓ handle_lookup_order")
print("✓ handle_process_refund")
print("✓ handle_escalate_to_human")

## Step 3. Claude Agent SDK の @tool デコレーターでツールを定義する

ビジネスロジック関数を `@tool` デコレーターでラップし、Agent SDK が理解できる形式にします。

In [ ]:
@tool(
    "get_customer",
    "顧客IDから本人確認済みの顧客プロフィールを取得する。返金・注文照会の前提確認に使う。",
    {"customer_id": str},
)
async def get_customer_tool(args: dict[str, Any]) -> dict[str, Any]:
    """顧客情報取得ツール"""
    payload = handle_get_customer(args["customer_id"])
    return build_tool_result(payload, is_error=payload.get("isError", False))


@tool(
    "lookup_order",
    "注文IDで完全な注文詳細を取得する。商品明細、配送状態、返金可否を確認したい時に使う。",
    {"order_id": str},
)
async def lookup_order_tool(args: dict[str, Any]) -> dict[str, Any]:
    """注文詳細取得ツール"""
    payload = handle_lookup_order(args["order_id"])
    return build_tool_result(payload, is_error=payload.get("isError", False))


@tool(
    "process_refund",
    "返金可能な注文に対して返金を実行する。副作用があり、高額返金は人間承認へエスカレーションする。",
    {"order_id": str, "amount": float, "reason": str},
)
async def process_refund_tool(args: dict[str, Any]) -> dict[str, Any]:
    """返金処理ツール"""
    payload = handle_process_refund(
        args["order_id"],
        args["amount"],
        args["reason"],
    )
    return build_tool_result(payload, is_error=payload.get("isError", False))


@tool(
    "escalate_to_human",
    "複雑なケースや高額返金など、人間の判断が必要な場合に使用する。",
    {"reason": str, "context": dict},
)
async def escalate_to_human_tool(args: dict[str, Any]) -> dict[str, Any]:
    """人間エスカレーションツール"""
    payload = handle_escalate_to_human(args["reason"], args["context"])
    return build_tool_result(payload, is_error=False)


section("Agent SDK ツールの定義完了")
print("✓ @tool get_customer")
print("✓ @tool lookup_order")
print("✓ @tool process_refund")
print("✓ @tool escalate_to_human")

## Step 4. PreToolUse フックで高額返金をブロックする

ビジネスルールをコードで強制するため、PreToolUse フックを実装します。

In [ ]:
async def pre_tool_use_refund_check(
    input_data: HookInput,
    tool_use_id: str | None,
    context: HookContext,
) -> HookJSONOutput:
    """返金処理前に金額をチェックし、閾値を超える場合はブロックする"""
    tool_input = input_data.get("tool_input", {})
    amount = tool_input.get("amount", 0)
    
    if amount > REFUND_THRESHOLD:
        print(f"🚫 返金額 {amount} 円が閾値 {REFUND_THRESHOLD} 円を超えています")
        return {
            "systemMessage": f"🚫 返金処理はポリシーによりブロックされました (金額: {amount} 円)",
            "reason": f"高額返金 ({amount} 円) は人手確認が必要です。escalate_to_human を使用してください。",
            "hookSpecificOutput": {
                "permissionDecision": "deny",
                "blockedAmount": amount,
                "threshold": REFUND_THRESHOLD,
                "suggestedAction": "escalate_to_human",
            },
        }
    
    print(f"✓ 返金額 {amount} 円は閾値内です")
    return {}


section("PreToolUse フックの定義完了")
print("✓ pre_tool_use_refund_check (高額返金ガード)")

## Step 5. PostToolUse フックでツール結果を正規化する

ツール実行後に結果を監査・正規化するフックを実装します。

In [ ]:
async def post_tool_use_audit(
    input_data: HookInput,
    tool_use_id: str | None,
    context: HookContext,
) -> HookJSONOutput:
    """ツール実行後に監査ログを記録する"""
    tool_name = input_data.get("tool_name", "unknown")
    tool_input = input_data.get("tool_input", {})
    
    print(f"📝 監査ログ: {tool_name} が実行されました")
    print(f"   入力: {json.dumps(tool_input, ensure_ascii=False)}")
    
    return {
        "reason": "ツール実行を監査しました",
    }


section("PostToolUse フックの定義完了")
print("✓ post_tool_use_audit (監査ログ)")

## Step 6. MCP サーバーを作成し、エージェントを実行する

すべてのツールを MCP サーバーに登録し、ClaudeSDKClient で実行します。

In [ ]:
async def run_support_agent(user_message: str) -> str:
    """サポートエージェントを実行する"""
    # MCP サーバーを作成してツールを登録
    server = create_sdk_mcp_server(
        name="support",
        version="1.0.0",
        tools=[
            get_customer_tool,
            lookup_order_tool,
            process_refund_tool,
            escalate_to_human_tool,
        ],
    )
    
    # システムプロンプト
    system_prompt = """
あなたは顧客サポートエージェントです。

## 基本方針
1. 必ず最初に get_customer で顧客情報を確認してください
2. 注文に関する問い合わせには lookup_order で詳細を確認してください
3. 返金リクエストには慎重に対応し、必要に応じて escalate_to_human を使用してください

## 重要なルール
- 高額返金 (10,000円超) は自動処理できません。escalate_to_human を使用してください
- エラーが発生した場合は、ユーザーに分かりやすく説明してください
- 構造化エラーの isRetryable フラグを確認し、適切に対応してください
""".strip()
    
    # エージェントオプションを設定
    options = ClaudeAgentOptions(
        system_prompt=system_prompt,
        max_turns=10,
        mcp_servers={"support": server},
        allowed_tools=[
            "mcp__support__get_customer",
            "mcp__support__lookup_order",
            "mcp__support__process_refund",
            "mcp__support__escalate_to_human",
        ],
        hooks={
            "PreToolUse": [
                HookMatcher(
                    matcher="mcp__support__process_refund",
                    hooks=[pre_tool_use_refund_check],
                )
            ],
            "PostToolUse": [
                HookMatcher(
                    matcher="*",
                    hooks=[post_tool_use_audit],
                )
            ],
        },
    )
    
    final_response = ""
    
    # エージェントを実行
    async with ClaudeSDKClient(options=options) as client:
        await client.query(user_message)
        
        async for message in client.receive_response():
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, TextBlock):
                        print(f"\n🤖 Claude: {block.text}")
                        final_response = block.text
                    elif isinstance(block, ToolUseBlock):
                        print(f"\n🔧 ツール呼び出し: {block.name}")
                        print(f"   引数: {json.dumps(block.input, ensure_ascii=False)}")
            elif isinstance(message, ResultMessage):
                if message.result and not final_response:
                    final_response = message.result
    
    return final_response


section("エージェント実行関数の定義完了")
print("✓ run_support_agent")

## Step 7. シナリオ1: 通常の注文確認 (低額)

In [ ]:
import asyncio

# セッション状態をリセット
session_state["customer_verified"] = False
session_state["verified_customer_id"] = None

section("シナリオ1: 通常の注文確認と返金")
result = await run_support_agent(
    "顧客ID CUST-001 です。注文 ORD-002 について確認して、5,000円の返金をお願いします。"
)
print(f"\n最終レスポンス: {result}")

### 確認ポイント

1. エージェントが自動的に `get_customer` → `lookup_order` → `process_refund` の順で実行したか
2. PreToolUse フックが金額をチェックし、閾値内のため承認したか
3. PostToolUse フックが各ツール実行を監査ログに記録したか

## Step 8. シナリオ2: 高額返金でエスカレーション

In [ ]:
# セッション状態をリセット
session_state["customer_verified"] = False
session_state["verified_customer_id"] = None

section("シナリオ2: 高額返金でエスカレーション")
result = await run_support_agent(
    "顧客ID CUST-001 です。注文 ORD-001 について、100,000円の返金をお願いします。"
)
print(f"\n最終レスポンス: {result}")

### 確認ポイント

1. PreToolUse フックが高額返金を検出し、`process_refund` の実行をブロックしたか
2. エージェントがブロックメッセージを理解し、代わりに `escalate_to_human` を呼び出したか
3. エスカレーション結果に適切なコンテキスト情報が含まれているか

## まとめ

この演習では、以下を学びました:

1. **@tool デコレーター**: ビジネスロジックを Agent SDK のツールとして定義
2. **create_sdk_mcp_server()**: ツールを MCP サーバーに登録
3. **ClaudeSDKClient**: エージェントの実行とメッセージストリーミング
4. **PreToolUse フック**: ツール実行前のビジネスルールチェック
5. **PostToolUse フック**: ツール実行後の監査と正規化
6. **構造化エラー**: エラーカテゴリ、リトライ可否、ユーザーメッセージの設計
7. **セッション状態**: プログラム的な前提条件ゲートの実装

## 完成版 Lab 参照

より詳細な実装とアンチパターンの解説は、完成版 Lab を参照してください:
- [../labs/01-support-agent/](../labs/01-support-agent/)
- Claude Agent SDK 公式ドキュメント: https://platform.claude.com/docs/en/agent-sdk/overview